# TTS Engine Prototyping
It is a pain to restart the API Server since we need to unload/load the engine every time.

In [16]:
from queue import Queue
from RealtimeTTS import TextToAudioStream, CoquiEngine

In [17]:
audio_queue = Queue()
chunks_received = 0
engine = None
engine_stream = None
pending_feed = Queue()

def load_engine(model_name="xtts_v2", voice="voices/lance.wav"):
    global engine, engine_stream, pending_feed, audio_queue, chunks_received
    audio_queue = Queue()
    chunks_received = 0
    engine = CoquiEngine(model_name=model_name, voice=voice)
    engine_stream = TextToAudioStream(engine, on_audio_stream_stop=on_audio_stream_stop)
    pending_feed = Queue() 


def on_audio_stream_stop():
    print("Audio stream stopped")
    global chunk_queue, pending_feed
    chunk_queue.put(200)
    pending_feed = Queue()


In [18]:
load_engine()

In [19]:
import openai

In [20]:
client = openai.OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [21]:
# ? Temporarily Stripped froom tts_api/runner.py
def openai_generator(gen_stream):
    """Generator for OpenAI streaming API.
    Yields sentences as they are completed."""
    payload = ""
    for chunk in gen_stream:
        if (content := chunk.choices[0].delta.content) is not None:
            # print(content, end="-")
            ends = ["?", ".", "!"]
            results = [end in content for end in ends]
            if any(results):
                payload += content
                # Get index of last found end in content
                last = max([payload.rindex(ends[i]) for i, x in enumerate(results) if x])
                feed, payload = payload[:last+1], payload[last+1:]
                # print("EOL")
                yield feed
            else:
                payload += content

    if payload.strip() != "":
        yield payload

In [ ]:
from fastapi import FastAPI
import fastapi
from contextlib import asynccontextmanager
import uvicorn
import io
import asyncio
from queue import Queue
app = FastAPI()
queue = Queue()
async def read_websocket(websocket: fastapi.WebSocket):
    global finished
    try:
        while True:
            data = await websocket.receive_text()
            if data == "END":
                break
            print(f"{data}", end="")
            queue.put(data)
    except fastapi.websockets.WebSocketDisconnect:
        print("Client disconnected.")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        queue.put(200)


chunk_queue = Queue()
async def send_to_ws(websocket: fastapi.WebSocket):
    global chunk_queue
    try:
        while True:
            chunk = None
            try:
                chunk = chunk_queue.get_nowait()
            except Exception as e: pass
            if chunk == 200:
                break
            
            if chunk is None:
                await asyncio.sleep(0.01)
                continue
            await websocket.send_bytes(chunk)
    except fastapi.websockets.WebSocketDisconnect:
        print("Client disconnected.")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        print("Send to ws task ended.")
        if websocket.client_state != fastapi.websockets.WebSocketState.DISCONNECTED:
            await websocket.send_text("END")
def on_audio_chunk(chunk):
    global chunk_queue
    chunk_queue.put(chunk)

def handle_ws():
    while True:
        var = queue.get(block=True)
        if var == 200:
            break

        yield var

@app.websocket("/ws")
async def websocket_endpoint(websocket: fastapi.WebSocket):
    global engine_stream

    await websocket.accept()
    print("Client connected.")
    send_to_ws_task = asyncio.create_task(send_to_ws(websocket))
    task = asyncio.create_task(read_websocket(websocket))
    engine_stream.feed(handle_ws()).play_async(muted=True, on_audio_chunk=on_audio_chunk)
    
    await asyncio.gather(task, send_to_ws_task)
    # Clear chunk_queue
    while not chunk_queue.empty():
        chunk_queue.get()
    print("Client disconnected, chunks cleared.")
    

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("API started.")
    yield
    print( "API stopped.")


if __name__ == "__main__":
    config = uvicorn.Config(app)
    server = uvicorn.Server(config)
    await server.serve()

INFO:     Started server process [9596]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     ('127.0.0.1', 49815) - "WebSocket /ws" [accepted]
INFO:     connection open


Client connected.
Once upon a time, there was a courageous knight who embarked on a perilous journey to save a fair maiden from a dragon's clutches, eventually slaying the beast and securing her freedom. Inspirational tales like this continue to captivate our imagination.

INFO:     connection closed


Audio stream stopped
Send to ws task ended.
Client disconnected.


INFO:     ('127.0.0.1', 49835) - "WebSocket /ws" [accepted]
INFO:     connection open


...

INFO:     ('127.0.0.1', 49852) - "WebSocket /ws" [accepted]
INFO:     connection open


Client connected.
Once upon a time, a curious little girl named Lucy discovered a secret garden. In the garden, hidden among the vibrant flowers, was a magical ancient tree. It sparkled like a star, and Lucy found her new special friend - an enchanting talking squirrel named Quiddle.

INFO:     connection closed


Audio stream stopped
Send to ws task ended.
Client disconnected.


INFO:     connection closed
INFO:     Shutting down
INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)
